# 18장 · 저장된 색인 다시 열고 FAISS와 비교하기

매일 여섯 쪽을 다시 임베딩하면 같은 자료에 호출 비용이 반복됩니다. 오늘은 어제 만든 색인의 이름과 저장 위치를 확인하고, 새 커널에서 동일 레코드를 다시 엽니다. 마지막에는 벡터 검색 부분만 FAISS로 옮겨 두 도구의 결과를 비교합니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다. FAISS 비교에 `%pip install faiss-cpu numpy`가 필요합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 메모리에서 사라지는 것과 디스크에 남는 것

노트북의 index 변수는 커널을 닫으면 사라집니다. Chroma의 PersistentClient가 쓰는 폴더에는 문서와 벡터가 남습니다. 수업 함수는 모델명·문단 ID·텍스트로 컬렉션 이름을 정합니다. 파일 내용이 바뀌면 다른 컬렉션을 열므로 예전 벡터와 새 자료가 섞이지 않습니다.

In [2]:
from rag_lab import open_index, retrieve
index = open_index()
collection, encoder = index
print("컬렉션", collection.name, "저장한 쪽", collection.count())
question = "2027년 3월 A형 녹음 파일은 몇 분마다 나뉘나요?"
print([r["id"] for r in retrieve(index, question, k=3)])

컬렉션 recorder-2bfb240a53a6e8ec 저장한 쪽 6
['recorder-a-1.0.pdf#p2', 'recorder-a-1.1.pdf#p2', 'recorder-b-1.0.pdf#p2']


방금 나온 이름·쪽수·상위 ID를 메모합니다. 커널을 재시작하고 준비 셀부터 여기까지 다시 실행하세요. 저장 폴더와 문서가 같으면 기존 문서 임베딩을 가져옵니다. 새 질문의 벡터는 검색할 때 별도로 만듭니다. 빈 폴더에서 처음 색인을 만드는 실행과 이미 만든 색인을 여는 실행을 구분하세요.

## 다른 검색 엔진에는 무엇을 넘길까

FAISS는 벡터의 위치를 정수로 반환합니다. 정수 0이 어느 PDF인지는 입력 순서를 기록한 사람이 연결해야 합니다. 같은 값으로 공정하게 비교하려고 Chroma가 보관한 벡터를 그대로 꺼내고, 질문도 한 번만 임베딩합니다. 두 도구 모두 여기서는 제곱 L2 거리(두 벡터 좌표 차이 제곱의 합)가 작은 순서로 찾습니다.

In [3]:
import faiss
import numpy as np

stored = collection.get(include=["embeddings"])
matrix = np.asarray(stored["embeddings"], dtype="float32")
engine = faiss.IndexFlatL2(matrix.shape[1])  # 각 벡터의 숫자 개수를 알려 줍니다.
engine.add(matrix)
qv = np.asarray([encoder.embed_query(question)], dtype="float32")
distances, positions = engine.search(qv, 3)
print("FAISS", [stored["ids"][i] for i in positions[0]])
found = collection.query(query_embeddings=qv.tolist(), n_results=3)
print("Chroma", found["ids"][0])

FAISS ['recorder-a-1.0.pdf#p2', 'recorder-a-1.1.pdf#p2', 'recorder-b-1.0.pdf#p2']
Chroma ['recorder-a-1.0.pdf#p2', 'recorder-a-1.1.pdf#p2', 'recorder-b-1.0.pdf#p2']


여섯 쪽에서 순서가 같다면 같은 벡터와 거리를 사용한 결과입니다. 모든 자료 규모에서 항상 같다는 뜻으로 확장하지 마세요. 후보 개수, 동점, 거리 방식, 근사 검색 설정이 달라지면 순서를 따로 확인해야 합니다. 문서 ID를 보지 않고 배열의 첫째 줄끼리만 비교하면 순서를 뒤바꾼 버그도 놓칩니다.

## 벡터 번호와 문서 이름의 대응이 틀어지는 경우

두 도구가 다른 문서를 돌려주는 것처럼 보일 때 먼저 입력 행과 출처 ID가 같은 순서였는지 봅니다. 아래에서는 거리도 검색 결과도 그대로 두고 문서 이름만 잘못 연결합니다. 학습 예제라서 불일치가 드러나지만 실제 앱에서는 그럴듯한 다른 쪽을 인용할 수 있습니다.

In [4]:
correct_ids = stored["ids"]
shifted_ids = correct_ids[1:] + correct_ids[:1]
pos = int(positions[0][0])
print("같은 벡터 위치", pos, "거리", round(float(distances[0][0]), 3))
print("올바른 문서", correct_ids[pos])
print("이름 목록만 민 경우", shifted_ids[pos])

같은 벡터 위치 1 거리 0.865
올바른 문서 recorder-a-1.0.pdf#p2
이름 목록만 민 경우 recorder-a-1.1.pdf#p1


벡터 위치와 거리는 같은데 인용한 파일이 달라졌습니다. 엔진을 바꾸기 앞서 ID와 벡터 행을 한 묶음으로 저장했는지 확인하면 검색 알고리즘을 잘못 의심하는 시간을 줄일 수 있습니다. 작은 거리 차이만 보고 어느 엔진이 더 정확하다고 결론짓지도 마세요. 사람의 정답 쪽과 대조하는 과정은 22장에서 수행합니다.

거리 함수가 바뀌면 두 목록의 숫자를 그대로 대조할 수 없습니다. 예를 들어 작은 값이 앞서는 거리와 큰 값이 앞서는 유사도를 모두 “점수”라고 적으면 낮은 쪽을 실수로 탈락시킬 수 있습니다. 여기서는 같은 질문 벡터의 같은 L2 설정으로 문단 ID를 먼저 비교합니다. [FAISS 기본 검색 참고](https://github.com/facebookresearch/faiss/wiki/Getting-started)

## 실습 · 새 문단을 하나만 바꾼 색인 구분하기

전체 자료를 다시 읽고 메모리 속 복사본 첫 페이지에 “검수 표시” 한 줄을 붙입니다. 새 컬렉션 이름과 이전 컬렉션의 쪽수를 기록하세요. 수업 PDF 원본은 보존하세요. 짝에게 원래 색인으로 다시 열어 같은 질문을 검색하도록 요청합니다.

In [5]:
# 원래 pages를 복사해 첫 쪽의 text만 교체합니다.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>from rag_lab import MANUALS, read_pages
original = [p for file in MANUALS for p in read_pages(file)]
changed = [{**p} for p in original]
changed[0][&quot;text&quot;] += &quot;\n검수 표시&quot;
other, _ = open_index(changed)
print(collection.name, other.name, collection.count(), other.count())</code></pre><p>원본과 수정본은 서로 다른 컬렉션 이름과 각각 6개의 쪽을 갖습니다. 다음 실습에 쓸 색인은 원본입니다. 색인 이름이 다르면 이전 수업에서 적은 검색 결과를 그대로 비교할 수 없습니다.</p><p>집필 시 실행 결과</p><pre>recorder-2bfb240a53a6e8ec recorder-ae5547dba792836b 6 6</pre></details>

## 독립 실습 · 다른 사람이 같은 색인을 열 수 있는가

작업 폴더, 컬렉션 이름, 문서 수, 임베딩 모델명, 질문, 상위 세 ID를 적습니다. 짝은 커널을 재시작한 뒤 같은 폴더에서 다시 열어 결과를 비교합니다. 다음에는 저장 경로만 다른 빈 폴더로 바꿉니다. 결과가 같더라도 이전 색인을 재사용했다고 판단할 수 있는지, 어느 폴더에 자료가 생겼는지 설명하세요.

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>같은 저장 경로·컬렉션에서 6개 레코드가 이미 존재한다면 공용 함수는 기존 문서를 다시 임베딩하지 않습니다. 빈 경로에는 새 데이터베이스가 생깁니다. 모델·자료가 같으면 두 색인의 이름과 문서 수도 같을 수 있습니다. 이름·순위만으로 같은 파일을 열었다고 판단하지 말고 실행한 경로와 파일 생성 여부를 같이 봅니다.</p></details>

## 다음 수업에 가져갈 것

기준 컬렉션 이름, PDF 세 파일, 질문, 상위 ID를 기록합니다. 다음 장은 검색 순위에서 끝나지 않고 같은 원래 질문을 근거 문단과 함께 모델에 전달합니다.